# Linear regression from scratch

Fit a line, inspect full-batch gradient descent, and compare both solvers to scikit-learn.
All data are generated locally. Run after `python -m pip install -e '.[dev]'` from the repository.

The objective is $J=(\|Xw+b-y\|^2+\alpha\|w\|^2)/(2n)$.
The intercept is not regularized. See [the math note](../docs/linear_regression.md).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression as SklearnLinearRegression
from sklearn.linear_model import Ridge

from minilearn import LinearRegression
from minilearn.metrics import mean_squared_error
from minilearn.utils import StandardScaler, train_test_split

rng = np.random.default_rng(42)
X = rng.uniform(-3, 3, size=(160, 1))
y = 2.5 * X[:, 0] + 1.2 + rng.normal(0, 0.5, len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler().fit(X_train)
train_scaled = scaler.transform(X_train)
test_scaled = scaler.transform(X_test)

## Fit without leaking test information

Fit scaling statistics only on the training split. Both reference and MiniLearn
models see the same scaled features, so the ridge penalties are comparable.

In [ ]:
normal = LinearRegression().fit(train_scaled, y_train)
gd = LinearRegression(solver="gd", learning_rate=0.1, tol=1e-10, random_state=42).fit(
    train_scaled, y_train
)
reference = SklearnLinearRegression().fit(train_scaled, y_train)
assert gd.converged_
np.testing.assert_allclose(normal.predict(test_scaled), reference.predict(test_scaled), atol=1e-8)
np.testing.assert_allclose(gd.predict(test_scaled), reference.predict(test_scaled), atol=1e-8)
print(f"Gradient descent converged in {gd.n_iter_} updates.")

In [ ]:
grid = np.linspace(-3, 3, 200).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(X_train[:, 0], y_train, s=15, alpha=0.5, label="Train")
axes[0].scatter(X_test[:, 0], y_test, s=20, label="Test")
axes[0].plot(grid[:, 0], normal.predict(scaler.transform(grid)), label="Least squares")
axes[0].plot(grid[:, 0], gd.predict(scaler.transform(grid)), "--", label="Gradient descent")
axes[0].set(xlabel="Feature", ylabel="Target", title="Fitted lines")
axes[0].legend()
axes[1].plot(np.arange(len(gd.loss_curve_)), gd.loss_curve_)
axes[1].set(xlabel="Updates", ylabel="Objective J", title="Gradient-descent convergence")
axes[1].set_yscale("log")
fig.tight_layout()
plt.show()

## Ridge comparison

`alpha=2.5` shrinks the slope; the intercept remains unpenalized. The table reports
held-out MSE and R² plus maximum coefficient differences. These are correctness
comparisons on synthetic data, not runtime benchmarks.

In [ ]:
print(f"{'Model':<18} {'Test MSE':>12} {'Test R2':>12} {'Max coef delta':>16}")
for alpha in (0.0, 2.5):
    ref = (Ridge(alpha=alpha) if alpha else SklearnLinearRegression()).fit(train_scaled, y_train)
    for solver in ("normal", "gd"):
        model = LinearRegression(
            solver=solver, alpha=alpha, learning_rate=0.1, tol=1e-10, random_state=42
        ).fit(train_scaled, y_train)
        predicted = model.predict(test_scaled)
        np.testing.assert_allclose(predicted, ref.predict(test_scaled), atol=1e-8)
        delta = np.max(np.abs(model.coef_ - ref.coef_))
        label = f"{solver}, alpha={alpha}"
        print(
            f"{label:<18} {mean_squared_error(y_test, predicted):12.6f} "
            f"{model.score(test_scaled, y_test):12.6f} {delta:16.3e}"
        )

## What to try next

Change the noise level or L2 penalty. Try an overly large learning rate, observe
nonconvergence or divergence, and relate it to the Hessian eigenvalues in the math
note. Next algorithm: binary logistic regression, with numerical gradient checks.